# Lab 07 - Decision Trees
**Abdul Hadi Saqib | 467626**

In [1]:
import random
import math
from collections import Counter

In [2]:
attributes = ["Alt", "Bar", "Fri", "Hun", "Pat", "Price", "Rain", "Res", "Type", "Est"]

dataset = [
    {"Alt": "Yes", "Bar": "No",  "Fri": "No",  "Hun": "Yes", "Pat": "Some", "Price": "$$$", "Rain": "No",  "Res": "Yes", "Type": "French",  "Est": "0-10",  "WillWait": "Yes"},
    {"Alt": "Yes", "Bar": "No",  "Fri": "No",  "Hun": "Yes", "Pat": "Full", "Price": "$",   "Rain": "No",  "Res": "No",  "Type": "Thai",    "Est": "30-60", "WillWait": "No"},
    {"Alt": "No",  "Bar": "Yes", "Fri": "No",  "Hun": "No",  "Pat": "Some", "Price": "$",   "Rain": "No",  "Res": "No",  "Type": "Burger",  "Est": "0-10",  "WillWait": "Yes"},
    {"Alt": "Yes", "Bar": "No",  "Fri": "Yes", "Hun": "Yes", "Pat": "Full", "Price": "$",   "Rain": "Yes", "Res": "No",  "Type": "Thai",    "Est": "10-30", "WillWait": "Yes"},
    {"Alt": "Yes", "Bar": "No",  "Fri": "Yes", "Hun": "No",  "Pat": "Full", "Price": "$$$", "Rain": "No",  "Res": "Yes", "Type": "French",  "Est": ">60",   "WillWait": "No"},
    {"Alt": "No",  "Bar": "Yes", "Fri": "No",  "Hun": "Yes", "Pat": "Some", "Price": "$$",  "Rain": "Yes", "Res": "Yes", "Type": "Italian", "Est": "0-10",  "WillWait": "Yes"},
    {"Alt": "No",  "Bar": "Yes", "Fri": "No",  "Hun": "No",  "Pat": "None", "Price": "$",   "Rain": "Yes", "Res": "No",  "Type": "Burger",  "Est": "0-10",  "WillWait": "No"},
    {"Alt": "No",  "Bar": "No",  "Fri": "No",  "Hun": "Yes", "Pat": "Some", "Price": "$$",  "Rain": "Yes", "Res": "Yes", "Type": "Thai",    "Est": "0-10",  "WillWait": "Yes"},
    {"Alt": "No",  "Bar": "Yes", "Fri": "Yes", "Hun": "No",  "Pat": "Full", "Price": "$",   "Rain": "Yes", "Res": "No",  "Type": "Burger",  "Est": ">60",   "WillWait": "No"},
    {"Alt": "Yes", "Bar": "Yes", "Fri": "Yes", "Hun": "Yes", "Pat": "Full", "Price": "$$$", "Rain": "No",  "Res": "Yes", "Type": "Italian", "Est": "10-30", "WillWait": "No"},
    {"Alt": "No",  "Bar": "No",  "Fri": "No",  "Hun": "No",  "Pat": "None", "Price": "$",   "Rain": "No",  "Res": "No",  "Type": "Thai",    "Est": "0-10",  "WillWait": "No"},
    {"Alt": "Yes", "Bar": "Yes", "Fri": "Yes", "Hun": "Yes", "Pat": "Full", "Price": "$",   "Rain": "No",  "Res": "No",  "Type": "Burger",  "Est": "30-60", "WillWait": "Yes"},
]

random.seed(42)
shuffled = dataset.copy()
random.shuffle(shuffled)
trainingSet = shuffled[:10]
validationSet = shuffled[10:]

print(f"Training set size: {len(trainingSet)}")
print(f"Validation set size: {len(validationSet)}")
print(f"\nValidation examples:")
for ex in validationSet:
    print(f"  {ex}")

Training set size: 10
Validation set size: 2

Validation examples:
  {'Alt': 'Yes', 'Bar': 'No', 'Fri': 'No', 'Hun': 'Yes', 'Pat': 'Full', 'Price': '$', 'Rain': 'No', 'Res': 'No', 'Type': 'Thai', 'Est': '30-60', 'WillWait': 'No'}
  {'Alt': 'No', 'Bar': 'No', 'Fri': 'No', 'Hun': 'No', 'Pat': 'None', 'Price': '$', 'Rain': 'No', 'Res': 'No', 'Type': 'Thai', 'Est': '0-10', 'WillWait': 'No'}


In [3]:
def calcEntropy(examples):
    total = len(examples)
    if total == 0:
        return 0
    counts = Counter(ex["WillWait"] for ex in examples)
    entropy = 0.0
    for count in counts.values():
        p = count / total
        if p > 0:
            entropy -= p * math.log2(p)
    return entropy


def calcInformationGain(examples, attribute):
    totalEntropy = calcEntropy(examples)
    total = len(examples)
    values = set(ex[attribute] for ex in examples)
    weightedEntropy = 0.0
    for val in values:
        subset = [ex for ex in examples if ex[attribute] == val]
        weightedEntropy += (len(subset) / total) * calcEntropy(subset)
    return totalEntropy - weightedEntropy


def pluralityValue(examples):
    counts = Counter(ex["WillWait"] for ex in examples)
    return counts.most_common(1)[0][0]


def buildDecisionTree(examples, remainingAttributes, parentExamples):
    if not examples:
        return pluralityValue(parentExamples)

    labels = set(ex["WillWait"] for ex in examples)
    if len(labels) == 1:
        return labels.pop()

    if not remainingAttributes:
        return pluralityValue(examples)

    gains = {attr: calcInformationGain(examples, attr) for attr in remainingAttributes}
    bestAttribute = max(gains, key=gains.get)

    tree = {"attribute": bestAttribute, "branches": {}}
    values = set(ex[bestAttribute] for ex in examples)
    newAttributes = [a for a in remainingAttributes if a != bestAttribute]

    for val in values:
        subset = [ex for ex in examples if ex[bestAttribute] == val]
        subtree = buildDecisionTree(subset, newAttributes, examples)
        tree["branches"][val] = subtree

    return tree


print("Building decision tree...")
decisionTree = buildDecisionTree(trainingSet, attributes, trainingSet)
print("Decision tree built successfully!")

Building decision tree...
Decision tree built successfully!


In [4]:
def printTree(tree, indent=""):
    if isinstance(tree, str):
        print(f"{indent}=> {tree}")
        return
    attribute = tree["attribute"]
    for val, subtree in tree["branches"].items():
        print(f"{indent}[{attribute} = {val}]")
        printTree(subtree, indent + "    ")


printTree(decisionTree)

[Pat = Full]
    [Type = Burger]
        [Alt = No]
            => No
        [Alt = Yes]
            => Yes
    [Type = French]
        => No
    [Type = Thai]
        => Yes
    [Type = Italian]
        => No
[Pat = None]
    => No
[Pat = Some]
    => Yes


In [5]:
def classify(tree, example):
    if isinstance(tree, str):
        return tree
    attribute = tree["attribute"]
    val = example[attribute]
    if val in tree["branches"]:
        return classify(tree["branches"][val], example)
    return None


def evaluateTree(tree, testSet):
    correct = 0
    for ex in testSet:
        prediction = classify(tree, ex)
        actual = ex["WillWait"]
        isCorrect = prediction == actual
        print(f"Actual: {actual}, Predicted: {prediction}, Correct: {isCorrect}")
        if isCorrect:
            correct += 1
    print(f"\nCorrect predictions: {correct}/{len(testSet)}")
    print(f"Accuracy: {correct / len(testSet) * 100:.1f}%")
    return correct


print("=== Validation Set Results ===")
correctCount = evaluateTree(decisionTree, validationSet)

print("\n=== Training Set Results ===")
evaluateTree(decisionTree, trainingSet)

=== Validation Set Results ===
Actual: No, Predicted: Yes, Correct: False
Actual: No, Predicted: No, Correct: True

Correct predictions: 1/2
Accuracy: 50.0%

=== Training Set Results ===
Actual: Yes, Predicted: Yes, Correct: True
Actual: Yes, Predicted: Yes, Correct: True
Actual: Yes, Predicted: Yes, Correct: True
Actual: No, Predicted: No, Correct: True
Actual: No, Predicted: No, Correct: True
Actual: No, Predicted: No, Correct: True
Actual: Yes, Predicted: Yes, Correct: True
Actual: Yes, Predicted: Yes, Correct: True
Actual: No, Predicted: No, Correct: True
Actual: Yes, Predicted: Yes, Correct: True

Correct predictions: 10/10
Accuracy: 100.0%


10